In [189]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import CoolProp.CoolProp as CP
import thermo as thermo
from scipy.optimize import root_scalar

In [141]:
class Fluid:
    def __init__(self, fluid_type, T_C=None, P_Pa=None, m_dot=None, h=None):
        """
        fluid_type: string, e.g. 'Water', 'R134a'
        T_C: temperature in Celsius
        P_Pa: pressure in Pascals
        m_dot: mass flowrate in kg/s
        h: enthalpy in J/kg (optional, can be computed from T and P)
        """
        self.fluid_type = fluid_type
        self.T_C = T_C
        self.P_Pa = P_Pa
        self.m_dot = m_dot
        self.h = h

        self.update_state()

    def update_state(self):
        """Update the enthalpy and other properties if T and P (or h and P) are known."""
        try:
            if self.T_C is not None and self.P_Pa is not None:
                self.h = CP.PropsSI('H', 'T', self.T_C + 273.15, 'P', self.P_Pa, self.fluid_type)
            elif self.h is not None and self.P_Pa is not None:
                self.T_C = CP.PropsSI('T', 'H', self.h, 'P', self.P_Pa, self.fluid_type) - 273.15
        except Exception as e:
            print(f"Error updating state: {e}")

    def set_T(self, T_C):
        self.T_C = T_C
        self.update_state()

    def set_P(self, P_Pa):
        self.P_Pa = P_Pa
        self.update_state()

    def set_h(self, h):
        self.h = h
        self.update_state()

    def set_m_dot(self, m_dot):
        self.m_dot = m_dot

    def cp(self):
        """Return specific heat at current state (J/kg-K). Robust for all regions."""
        # Get saturated enthalpies at this pressure
        hL = CP.PropsSI('H', 'P', self.P_Pa, 'Q', 0, self.fluid_type)
        hV = CP.PropsSI('H', 'P', self.P_Pa, 'Q', 1, self.fluid_type)
        tol = 1e-3
        # Two-phase
        if hL < self.h < hV:
            return 1e12  # Effectively infinite
        # Single-phase: use PropsSI('C','H',h,'P',P,fluid) directly
        try:
            cp_val = CP.PropsSI('C', 'H', self.h, 'P', self.P_Pa, self.fluid_type)
        except Exception:
            # If still fails, likely at the edge, pick value at closest boundary
            if np.abs(self.h - hL) <= tol:
                cp_val = CP.PropsSI('C', 'Q', 0, 'P', self.P_Pa, self.fluid_type)
            elif np.abs(self.h - hV) <= tol:
                cp_val = CP.PropsSI('C', 'Q', 1, 'P', self.P_Pa, self.fluid_type)
            else:
                raise
        return cp_val


    def density(self):
        """Return density at current state (kg/m³)."""
        return CP.PropsSI('D', 'T', self.T_C + 273.15, 'P', self.P_Pa, self.fluid_type)

    def s(self):
        """Return entropy at current state (J/kg-K)."""
        return CP.PropsSI('S', 'T', self.T_C + 273.15, 'P', self.P_Pa, self.fluid_type)

    def as_dict(self):
        """Return a dictionary of current properties."""
        return {
            'fluid_type': self.fluid_type,
            'T_C': self.T_C,
            'P_Pa': self.P_Pa,
            'h': self.h,
            'm_dot': self.m_dot,
        }

    def __repr__(self):
        return f"<Fluid {self.fluid_type}: T={self.T_C:.2f} °C, P={self.P_Pa/1e5:.2f} bar, h={self.h/1e3:.2f} kJ/kg, m_dot={self.m_dot} kg/s>"

def is_two_phase(fluid):
    """Return True if the fluid is in the two-phase region at its current state."""
    hL = CP.PropsSI('H', 'P', fluid.P_Pa, 'Q', 0, fluid.fluid_type)
    hV = CP.PropsSI('H', 'P', fluid.P_Pa, 'Q', 1, fluid.fluid_type)
    return hL < fluid.h < hV


In [185]:
def chiller_heat_transfer(water_in, ref_in, UA_total, num_sub_hex, max_iter=30, tol=1e-4):
    
    UA_section = UA_total / num_sub_hex
    min_pinch = 1
    # Prepare lists for each station (N+1 points)
    hw = np.zeros(num_sub_hex+1)
    hr = np.zeros(num_sub_hex+1)
    hw[0] = CP.PropsSI('H', 'T', water_in.T_C + 273.15, 'P', water_in.P_Pa, water_in.fluid_type)
    hr[-1] = CP.PropsSI('H', 'T', ref_in.T_C + 273.15, 'P', ref_in.P_Pa, ref_in.fluid_type)
    # Guess initial profile: linear for both
    for i in range(1, num_sub_hex+1):
        hw[i] = hw[0]
    for i in range(num_sub_hex):
        hr[i] = hr[-1]
        
    for it in range(max_iter):
        print("iter:", it)
        hw_new = np.zeros_like(hw)
        hr_new = np.zeros_like(hr)
        hw_new = hw.copy()
        hr_new = hr.copy()
        #print(hw_new, hr_new)
        for i in range(num_sub_hex):
            # Section i couples water at i → i+1 with refrigerant at N-i → N-i-1
            w = Fluid(water_in.fluid_type, h=hw[i], P_Pa=water_in.P_Pa, m_dot=water_in.m_dot)
            r = Fluid(ref_in.fluid_type, h=hr[i], P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot)
            
            #fluid properties
            Cp_w = w.cp()
            Cp_r = r.cp()
            #print(w.T_C,Cp_w,r.T_C, Cp_r)
    
            
            #effectiveness-NTU setup
            Cw = w.m_dot * Cp_w
            Cr = r.m_dot * Cp_r
            Cmin = min(Cw, Cr) if min(Cw, Cr) > 0 else 1e-12
            Cmax = max(Cw, Cr) if max(Cw, Cr) > 0 else 1e-12
            NTU = UA_section / Cmin
            Crat = Cmin / Cmax
    
            # Effectiveness for counterflow
            if is_two_phase(r):
                eff = 1 - np.exp(-NTU)  # C_r infinite
                #print("2 phase eff")
            elif abs(Crat - 1) > 1e-6:
                eff = (1 - np.exp(-NTU * (1 - Crat))) / (1 - Crat * np.exp(-NTU * (1 - Crat)))
            else:
                eff = NTU / (1 + NTU)
            Qmax = Cmin * (w.T_C - r.T_C)
            Q = eff * Qmax
            h_r_max = Fluid(ref_in.fluid_type, T_C=w.T_C - min_pinch, P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot).h
            Q_max_pinch = r.m_dot * (h_r_max - hr_new[i])
            #print("Q: ", Q, "eff:", eff, "Cmin:", Cmin, "Cw:", Cw,"Cr:", Cr,"Qmax:", Qmax,"NTU:", NTU, "Qmax_pinch:", Q_max_pinch)
    
            # Update enthalpies for next node
            hw_new[i+1] = hw[i] - Q / w.m_dot
            hr_new[i+1] = hr[i] - Q / r.m_dot
            #print("T_w_in:", w.T_C, "T_w_out", Fluid(water_in.fluid_type, h=hw_new[i+1], P_Pa=water_in.P_Pa, m_dot=water_in.m_dot).T_C)
            #print("T_r_in:", r.T_C, "T_r_out", Fluid(ref_in.fluid_type, h=hr_new[i+1], P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot).T_C)
    
        # Check convergence (outlet enthalpies)
        max_diff_w = np.max(np.abs(hw_new - hw))
        max_diff_r = np.max(np.abs(hr_new - hr))
        if max_diff_w < tol and max_diff_r < tol:
            #print("break", hw_new, hw, hr_new, hr)
            break

        hw = hw_new.copy()
        hr = hr_new.copy()
        # Create Fluid lists for output
        water_states = [Fluid(water_in.fluid_type, h=hw[i], P_Pa=w.P_Pa, m_dot=w.m_dot) for i in range(num_sub_hex+1)]
        ref_states = [Fluid(ref_in.fluid_type, h=hr[i], P_Pa=r.P_Pa, m_dot=r.m_dot) for i in range(num_sub_hex+1)]
        tot_heat_trans = water_in.m_dot * (water_states[0].h - water_states[-1].h)
    return water_states, ref_states#, tot_heat_trans

def hx_with_bypass(water_in, ref_in, UA_total, num_sub_hex, bypass_ratio):
    # 1. Split water stream
    m_dot_total = water_in.m_dot
    m_dot_HX = (1 - bypass_ratio) * m_dot_total
    m_dot_bypass = bypass_ratio * m_dot_total
    
    # 2. Create Fluid objects for each stream
    water_in_HX = Fluid(water_in.fluid_type, T_C=water_in.T_C, P_Pa=water_in.P_Pa, m_dot=m_dot_HX)
    water_in_bypass = Fluid(water_in.fluid_type, T_C=water_in.T_C, P_Pa=water_in.P_Pa, m_dot=m_dot_bypass)
    
    # 3. Run the heat exchanger on the portion that goes through the HX
    water_states, ref_states = chiller_heat_transfer(water_in_HX, ref_in, UA_total, num_sub_hex)
    water_out_HX = water_states[-1]
    
    # 4. Mix the bypassed and cooled streams
    h_mixed = (m_dot_HX * water_out_HX.h + m_dot_bypass * water_in_bypass.h) / m_dot_total
    T_mixed = CP.PropsSI('T', 'H', h_mixed, 'P', water_in.P_Pa, water_in.fluid_type) - 273.15
    
    # 5. Create Fluid object for mixed output
    water_out_mixed = Fluid(water_in.fluid_type, T_C=T_mixed, P_Pa=water_in.P_Pa, m_dot=m_dot_total)
    water_out_mixed.set_h(h_mixed)  # ensure both T and h are consistent

    # 6. Print temperature profiles for water and refrigerant
    # 6. Print temperature and refrigerant quality profile
    print("\n--- Heat Exchanger Profile ---")
    print(f"{'Section':^8} | {'Water Temp (C)':^18} | {'Ref Temp (C)':^18} | {'Ref Quality':^12}")
    print("-" * 66)
    for i in range(len(water_states)):
        w = water_states[i]
        r = ref_states[len(ref_states)-1-i]  # counterflow
        try:
            Q = CP.PropsSI('Q', 'H', r.h, 'P', r.P_Pa, r.fluid_type)
        except Exception:
            Q = float('nan')
        print(f"{i:^8} | {w.T_C:^18.2f} | {r.T_C:^18.2f} | {Q:^12.3f}")
    print("-" * 66 + "\n")

    return water_out_mixed, water_out_HX, water_in_bypass

In [321]:
def chiller_heat_transfer_shooting(water_in, ref_in, UA_total, num_sub_hex, max_iter=50, tol=1e-5, verbose=False):
    
    UA_section = UA_total / num_sub_hex
    min_pinch = 0.01
    def step_through(h_r_out_guess):
        # Initialize enthalpy arrays
        hw = np.zeros(num_sub_hex+1)
        hr = np.zeros(num_sub_hex+1)
        hw[0] = CP.PropsSI('H', 'T', water_in.T_C + 273.15, 'P', water_in.P_Pa, water_in.fluid_type)
        hr[0] = h_r_out_guess
    
        for i in range(num_sub_hex):
            # Section i couples water at i → i+1 with refrigerant at N-i → N-i-1
            w = Fluid(water_in.fluid_type, h=hw[i], P_Pa=water_in.P_Pa, m_dot=water_in.m_dot)
            r = Fluid(ref_in.fluid_type, h=hr[i], P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot)
            
            #fluid properties
            Cp_w = w.cp()
            Cp_r = r.cp()
            #print(w.T_C,Cp_w,r.T_C, Cp_r)
            #effectiveness-NTU setup
            Cw = w.m_dot * Cp_w
            Cr = r.m_dot * Cp_r
            Cmin = min(Cw, Cr) if min(Cw, Cr) > 0 else 1e-12
            Cmax = max(Cw, Cr) if max(Cw, Cr) > 0 else 1e-12
            NTU = UA_section / Cmin
            Crat = Cmin / Cmax
    
            # Effectiveness for counterflow
            if is_two_phase(r):
                eff = 1 - np.exp(-NTU)  # C_r infinite
                #print("2 phase eff")
            elif abs(Crat - 1) > 1e-6:
                eff = (1 - np.exp(-NTU * (1 - Crat))) / (1 - Crat * np.exp(-NTU * (1 - Crat)))
            else:
                eff = NTU / (1 + NTU)
            Qmax = Cmin * (w.T_C - r.T_C)
            Q = eff * Qmax
            h_r_max = Fluid(ref_in.fluid_type, T_C=w.T_C - min_pinch, P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot).h
            #Q_max_pinch = r.m_dot * (h_r_max - hr_new[i])
            #print("Q: ", Q, "eff:", eff, "Cmin:", Cmin, "Cw:", Cw,"Cr:", Cr,"Qmax:", Qmax,"NTU:", NTU)
    
            # Update enthalpies for next node
            hw[i+1] = hw[i] - Q / w.m_dot
            hr[i+1] = hr[i] - Q / r.m_dot
            #print("T_w_in:", w.T_C, "T_w_out", Fluid(water_in.fluid_type, h=hw_new[i+1], P_Pa=water_in.P_Pa, m_dot=water_in.m_dot).T_C)
            #print("T_r_in:", r.T_C, "T_r_out", Fluid(ref_in.fluid_type, h=hr_new[i+1], P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot).T_C)
        return hr[-1], hw, hr  # predicted refrigerant inlet enthalpy, plus profiles

    # Physical bounds for refrigerant outlet enthalpy guess
    #hV = CP.PropsSI('H', 'P', ref_in.P_Pa, 'Q', 1, ref_in.fluid_type)
    hL = CP.PropsSI('H', 'P', ref_in.P_Pa, 'Q', 0, ref_in.fluid_type)
    h_r_out_low = hL
    T_r_out_high = water_in.T_C - min_pinch
    h_r_out_high = CP.PropsSI('H', 'T', T_r_out_high + 273.15, 'P', ref_in.P_Pa, ref_in.fluid_type)
    
    def shooting_target(hr0_guess):
        try:
            h_r_in_pred, _, _ = step_through(hr0_guess)
            return h_r_in_pred - CP.PropsSI('H', 'T', ref_in.T_C + 273.15, 'P', ref_in.P_Pa, ref_in.fluid_type)
        except Exception as e:
            if verbose:
                print(f"Bad guess (h_r_out={hr0_guess}): {e}")
            return -1e6
    try:
        sol = root_scalar(shooting_target, bracket=[h_r_out_low, h_r_out_high], rtol=tol, maxiter=max_iter, method='bisect')
        if not sol.converged:
            raise RuntimeError("Shooting method did not converge")

        _, hw, hr = step_through(sol.root)
        water_states = [Fluid(water_in.fluid_type, h=hw[i], P_Pa=water_in.P_Pa, m_dot=water_in.m_dot) for i in range(num_sub_hex+1)]
        ref_states = [Fluid(ref_in.fluid_type, h=hr[i], P_Pa=ref_in.P_Pa, m_dot=ref_in.m_dot) for i in range(num_sub_hex+1)]
        tot_heat_trans = water_in.m_dot * (water_states[0].h - water_states[-1].h)
        return water_states, ref_states, tot_heat_trans
    except: #if breaks, likely because HX mismatch, impossible conditions
        if verbose:
            print("Invalid HX conditions, flow imbalance likely precludes given conditions")
        return water_states, ref_states, 0
        
    

def hx_with_bypass_shooting(water_in, ref_in, UA_total, num_sub_hex, bypass_ratio, verbose=False):
    # 1. Split water stream
    m_dot_total = ref_in.m_dot
    m_dot_HX = (1 - bypass_ratio) * m_dot_total
    m_dot_bypass = bypass_ratio * m_dot_total
    
    # 2. Create Fluid objects for each stream
    ref_in_HX = Fluid(ref_in.fluid_type, T_C=ref_in.T_C, P_Pa=ref_in.P_Pa, m_dot=m_dot_HX)
    ref_in_bypass = Fluid(ref_in.fluid_type, T_C=ref_in.T_C, P_Pa=ref_in.P_Pa, m_dot=m_dot_bypass)
    
    # 3. Run the heat exchanger on the portion that goes through the HX
    water_states, ref_states, Q_HX = chiller_heat_transfer_shooting(water_in, ref_in_HX, UA_total, num_sub_hex)
    ref_out_HX = ref_states[0]
    water_out_HX = water_states[-1]
    
    # 4. Mix the bypassed and cooled streams
    h_mixed = (m_dot_HX * ref_out_HX.h + m_dot_bypass * ref_in_bypass.h) / m_dot_total
    #T_mixed = CP.PropsSI('T', 'H', h_mixed, 'P', ref_in.P_Pa, ref_in.fluid_type) - 273.15
    
    # 5. Create Fluid object for mixed output
    ref_out_mixed = Fluid(ref_in.fluid_type, h=h_mixed, P_Pa=ref_in.P_Pa, m_dot=m_dot_total)
    #water_out_mixed.set_h(h_mixed)  # ensure both T and h are consistent

    # 6. Print temperature profiles for water and refrigerant
    # 6. Print temperature and refrigerant quality profile
    print("\n--- Heat Exchanger Profile ---")
    print(f"{'Section':^8} | {'Water Temp (C)':^18} | {'Ref Temp (C)':^18} | {'Ref Quality':^12}")
    print("-" * 66)
    for i in range(len(water_states)):
        w = water_states[i]
        r = ref_states[i]  # counterflow
        try:
            Q = CP.PropsSI('Q', 'H', r.h, 'P', r.P_Pa, r.fluid_type)
        except Exception:
            Q = float('nan')
        print(f"{i:^8} | {w.T_C:^18.2f} | {r.T_C:^18.2f} | {Q:^12.3f}")
    print("-" * 66 + "\n")
    print("Q_HX:",Q_HX)

    return ref_out_mixed, water_out_HX, Q_HX

def find_bypass_for_Q(water_in, ref_in, UA_total, num_sub_hex, Q_target, rtol, verbose):
    def obj(bypass_ratio):
        # Clamp bypass ratio to [0,1)
        if bypass_ratio < 0.0 or bypass_ratio >= 1.0:
            return 1e6
        try:
            _, _, Q_HX = hx_with_bypass_shooting(
                water_in, ref_in, UA_total, num_sub_hex, bypass_ratio, verbose=verbose
            )
            if Q_HX is None:
                print("invalid")
                return 0
            elif np.abs(Q_HX - Q_target) < Q_target * rtol:
                print("converged",bypass_ratio,Q_HX, Q_target)
                return 0
            else:
                print(bypass_ratio,Q_HX, Q_target)
                return Q_HX - Q_target
        except Exception as e:
            print(f"Bypass {bypass_ratio} invalid: {e}")
            return 1e6

    sol = root_scalar(obj, bracket=[0.0, 0.95], rtol=rtol, method='bisect')
    if not sol.converged:
        if verbose:
            print("Could not find a valid bypass ratio for the specified Q.")
        return None
    return sol.root

In [329]:
fluid_w = Fluid('water', T_C=37, P_Pa = 1e5, m_dot = 1.5)
fluid_ref = Fluid('R32', T_C=16, P_Pa = 18e5, m_dot = 2)
#chiller_heat_transfer_shooting(fluid_w, fluid_ref, 1e4,101)

#bypass_ratio = 0.5
#num_sub_hex = 101
#UA_total = 5e4
#ref_out_mixed, water_out_HX, Q_HX = hx_with_bypass_shooting(fluid_w, fluid_ref, UA_total, num_sub_hex, bypass_ratio)
#print("Mixed outlet ref:", ref_out_mixed)
#print("HX outlet water:", water_out_HX)
#print("Total heat transfer:", Q_HX)

Q_target = 40000
 
optimal_bypass = find_bypass_for_Q(fluid_w, fluid_ref, UA_total, num_sub_hex, Q_target, 1e-2, False)
if optimal_bypass is not None:
    print(f"Optimal bypass ratio for Q = {Q_target} is: {optimal_bypass:.4f}")
    ref_out_mixed, water_out_HX, Q_HX = hx_with_bypass_shooting(
        fluid_w, fluid_ref, UA_total, num_sub_hex, optimal_bypass
    )
    print(f"Achieved heat transfer: {Q_HX:.2f} W at bypass ratio {optimal_bypass:.4f}")
    print(f"Mixed water outlet temp: {water_out_mixed.T_C:.2f} °C")
else:
    print("No feasible bypass ratio found for the specified Q.")

Error updating state: unable to solve 1phase PY flash with Tmin=273.152, Tmax=372.767 due to error: HSU_P_flash_singlephase_Brent could not find a solution because Hmolar [-88.8142 J/mol] is below the minimum value of 1.1978218759 J/mol : PropsSI("T","H",-4929.942279,"P",100000,"water")
Error updating state: Saturation pressure [1.8e+06 Pa] corresponding to T [300.532 K] is within 1e-4 % of given p [1.8e+06 Pa] : PropsSI("H","T",300.5318814,"P",1800000,"R32")
Error updating state: Saturation pressure [1.8e+06 Pa] corresponding to T [300.532 K] is within 1e-4 % of given p [1.8e+06 Pa] : PropsSI("H","T",300.5318814,"P",1800000,"R32")
Error updating state: Saturation pressure [1.8e+06 Pa] corresponding to T [300.532 K] is within 1e-4 % of given p [1.8e+06 Pa] : PropsSI("H","T",300.5318814,"P",1800000,"R32")
Error updating state: unable to solve 1phase PY flash with Tmin=273.152, Tmax=372.767 due to error: HSU_P_flash_singlephase_Brent could not find a solution because Hmolar [-75.2623 J/m

ValueError: f(a) and f(b) must have different signs